#Build Races Dimension

 1. Read silver `races` table
 1. Read silver `circuits` table
 1. Join the data from `races` with `circuits` using `circuit_id`
 1. Select the required columns
     - races.season 
     - races.round 
     - races.race_name 
     - races.race_date 
     - circuits.circuit_name 
     - circuits.locality 
     - circuits.country
 1. Write the transformed data to gold `dim_races` table

![formula1-gold-data-erd.png](./formula1-gold-data-erd.png "formula1-gold-data-erd.png")

In [0]:
%run ../00-common/1.Environment-Config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

#### Step 1 - Read source tables
- `circuits`
- `races`

In [0]:
circuits_df = spark.table(f"{catalog_name}.{silver_schema}.circuits")
races_df = spark.table(f"{catalog_name}.{silver_schema}.races")

####Step 2 - Join `races` with `circuits` using `circuit_id`
Select the following columns  
  1. races.season 
  1. races.round 
  1. races.race_name 
  1. races.race_date 
  1. circuits.circuit_name 
  1. circuits.locality 
  1. circuits.country

In [0]:
dim_races_df = (
                races_df
                .join(
                    circuits_df,
                    races_df.circuit_id == circuits_df.circuit_id,
                    "inner"
                )
                .select(
                    races_df.season,
                    races_df.round,
                    races_df.race_name,
                    races_df.race_date,
                    circuits_df.circuit_name,
                    circuits_df.locality,
                    circuits_df.country
                )
)

In [0]:
display(dim_races_df)

In [0]:
(
    dim_races_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))